# 04: MNIST MLP にSVDをあてる

## 一言で言うと
**手書き数字を当てるニューラルネットを学習して、重みに対してSVDを充てる**ノート。  


In [1]:
# ============================================================
# ライブラリの import
# ============================================================
import torch
import torch.nn as nn          # 層の定義（Linear, ReLU など）
import torch.optim as optim    # 最適化（Adam など）

from torch.utils.data import DataLoader          # ミニバッチでデータを流す
from torchvision import datasets, transforms     # MNIST と前処理

In [2]:
# ============================================================
# デバイス選択（GPU があれば cuda、なければ cpu）
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
# 今の環境は CPU でも MNIST MLP なら数分で学習できる

device: cuda


In [3]:
# ============================================================
# MNIST データの準備
# ============================================================
# ToTensor(): 画像を [0,1] の float テンソルに変換
#   元: PIL Image (28×28) → 後: Tensor (1, 28, 28)
transform = transforms.Compose([
    transforms.ToTensor()
])

# root="./data" にダウンロード＆キャッシュされる
# 初回だけダウンロード。LeCun サイトが 404 でもミラーから取れるので正常
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# DataLoader: データをミニバッチに分割して渡す
train_loader = DataLoader(
    train_dataset,
    batch_size=64,   # 1回の更新で使う枚数
    shuffle=True     # 学習時は順番を混ぜる
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1000,
    shuffle=False    # 評価時は混ぜなくてよい
)

print("train size:", len(train_dataset))  # 60000（規定値）
print("test size:", len(test_dataset))    # 10000（規定値）

train size: 60000
test size: 10000


In [4]:
# ============================================================
# 1バッチだけ取り出して shape を確認
# ============================================================
images, labels = next(iter(train_loader))

print("images shape:", images.shape)  # (64, 1, 28, 28) = バッチ, チャネル, H, W
print("labels shape:", labels.shape)  # (64,)
print("labels:", labels[:10])         # 0〜9 のクラスラベル

images shape: torch.Size([64, 1, 28, 28])
labels shape: torch.Size([64])
labels: tensor([6, 0, 5, 1, 7, 7, 9, 7, 3, 3])


In [5]:
# ============================================================
# MLP の定義（圧縮前のベースライン構造）
# ============================================================
# 784 → 512 → 256 → 10
# あとで fc1 / fc2 の重みを SVD で圧縮する想定
class MNISTMLP(nn.Module):
    # 
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(784, 512)  # ここが圧縮の主な対象候補
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(512, 256)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(256, 10)   # 出力は 10 クラス（数字 0〜9）

    def forward(self, x):
        # x shape: (batch_size, 1, 28, 28)

        #-1→ 残りの次元を自動で計算する。
        # if self.training:
        # ...
        # else:
        # ...
        # 画像を1次元ベクトルに変換（MLP はベクトル入力）
        # x.size(0)→ バッチサイズ(64)をそのまま残す。
        x = x.view(x.size(0), -1)  # (batch_size, 784)

        x = self.fc1(x)
        x = self.relu1(x)

        x = self.fc2(x)
        x = self.relu2(x)

        x = self.fc3(x)  # logits（ソフトマックス前の生スコア）を返す

        return x

In [6]:
# ============================================================
# モデル・損失関数・最適化手法の準備
# ============================================================

# MNISTMLP() で、さっき定義したニューラルネット本体を作っています。
# .to(device) は、そのモデルをGPU に移しています。
model = MNISTMLP().to(device)

# 多クラス分類なので CrossEntropyLoss
# （内部で log_softmax + NLLLoss をやるので、モデル出力は logits のまま）
criterion = nn.CrossEntropyLoss()

# Adam: 学習率 0.001 はよく使う初期値
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

MNISTMLP(
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=256, out_features=10, bias=True)
)


In [7]:
# ============================================================
# 学習 1 epoch = 「全データを1回見て、間違いを直す」
# ============================================================
# 1バッチごとのやること（これが学習の本体）:
#   1. 画像を入れる → 予測が出る
#   2. 正解と比べて loss（間違い）を計算
#   3. 「どの重みをどう直せばいいか」を計算（backward）
#   4. 重みを少し更新（step）
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    # model.train() は nn.Module が持っている仕組み
    # モデル自身とその子モジュールの training 状態を True にします
    # 今はなくてもあっても変わらない
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:  # 64枚ずつ取り出す
        images = images.to(device)
        labels = labels.to(device)

        # PyTorch は勾配を足し込むので、毎回リセット
        optimizer.zero_grad()            # ① 前回の修正メモを消す

        # forwardを使って計算している
        # outputs = model.forward(images)と同じ
        outputs = model(images)          # ② 予測（10クラス分のスコア）

        loss = criterion(outputs, labels)  # ③ 間違いの大きさ

        # 誤差逆伝播をして勾配を作る
        # forwardで計算したlossをbackwardで勾配を計算する
        loss.backward()                  # ④ どこを直せばいいか計算

        # backwardで計算した勾配をstepで更新する
        optimizer.step()                 # ⑤ 重みを少し直す

        # loss.item() はそのバッチの平均損失
        # images.size(0) を掛けるのは、64枚分の損失の合計っぽいものに直して、
        # あとで全体平均を取るため
        total_loss += loss.item() * images.size(0)

        # 0~9での各スコアを比較して１番大きいものをとる→これが予測
        predicted = torch.argmax(outputs, dim=1)  # 一番高いスコアの数字

        # 予測と正解を比較して、正解している数を数える
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total   # 平均 loss
    accuracy = correct / total      # 正解率

    return avg_loss, accuracy

In [8]:
# ============================================================
# テストデータでの評価（学習はしない）
# ============================================================
def evaluate(model, test_loader, criterion, device):
    model.eval()  # 評価モード

    total_loss = 0.0
    correct = 0
    total = 0

    # 評価時は勾配不要 → メモリ節約・高速化
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [9]:
# ============================================================
# 学習ループ（5 epoch）
# ============================================================
# 目安: Test Acc が 97〜98% くらいまで上がれば OK
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f}, "
        f"Train Acc: {train_acc:.4f}, "
        f"Test Loss: {test_loss:.4f}, "
        f"Test Acc: {test_acc:.4f}"
    )

Epoch [1/5] Train Loss: 0.2352, Train Acc: 0.9303, Test Loss: 0.1320, Test Acc: 0.9590
Epoch [2/5] Train Loss: 0.0881, Train Acc: 0.9726, Test Loss: 0.1028, Test Acc: 0.9686
Epoch [3/5] Train Loss: 0.0590, Train Acc: 0.9814, Test Loss: 0.0780, Test Acc: 0.9747
Epoch [4/5] Train Loss: 0.0417, Train Acc: 0.9861, Test Loss: 0.0855, Test Acc: 0.9742
Epoch [5/5] Train Loss: 0.0322, Train Acc: 0.9895, Test Loss: 0.0693, Test Acc: 0.9815


In [10]:
# ============================================================
# パラメータ数を数える（圧縮後と比較する基準）
# ============================================================
# fc1: 784*512+512 = 401,920
# fc2: 512*256+256 = 131,328
# fc3: 256*10+10   =   2,570
# 合計             = 535,818
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

baseline_params = count_parameters(model)
print("Baseline params:", baseline_params)

Baseline params: 535818


In [11]:
# ============================================================
# 学習済み重みを保存（次のノートで読み込んで SVD 圧縮する）
# ============================================================
# state_dict = 各層の weight / bias だけを辞書で保存
torch.save(model.state_dict(), "mnist_mlp_baseline.pth")
print("saved: mnist_mlp_baseline.pth")

saved: mnist_mlp_baseline.pth
